# Classification sur la classe comestible
# Plus de donnée et donnée manquantes

Lien Kaggle: https://www.kaggle.com/competitions/playground-series-s4e8/data?select=test.csv <br>
Doc scikit: https://scikit-learn.org/stable/ <br>
Doc pandas: https://pandas.pydata.org/docs/

In [54]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder

from sklearn.compose import make_column_transformer

from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier

In [2]:
train=pd.read_csv('data_miss/train.csv', index_col='id')
test=pd.read_csv('data_miss/test.csv', index_col='id')

In [33]:
print(list(train.columns))
train.head()

['class', 'cap-diameter', 'cap-shape', 'cap-surface', 'cap-color', 'does-bruise-or-bleed', 'gill-attachment', 'gill-spacing', 'gill-color', 'stem-height', 'stem-width', 'stem-root', 'stem-surface', 'stem-color', 'veil-type', 'veil-color', 'has-ring', 'ring-type', 'spore-print-color', 'habitat', 'season']


,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
id,,,,,,,,,,,,,,,,,,,,,
0,e,8.80,f,s,u,f,a,c,w,4.51,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,a
1,p,4.51,x,h,o,f,a,c,n,4.79,...,NaN,y,o,NaN,NaN,t,z,NaN,d,w
2,e,6.94,f,s,b,f,x,c,w,6.85,...,NaN,s,n,NaN,NaN,f,f,NaN,l,w
3,e,3.88,f,y,g,f,s,NaN,g,4.16,...,NaN,NaN,w,NaN,NaN,f,f,NaN,d,u
4,e,5.85,x,l,w,f,d,NaN,w,3.37,...,NaN,NaN,w,NaN,NaN,f,f,NaN,g,a


In [72]:
quanti=list(train.select_dtypes(include=["float64"]).columns)
quali=list(train.select_dtypes(include=["object"]).drop(columns="class").columns)
train_y=train["class"]
train_x=train.drop(columns="class")


## Analyse des données

In [10]:
train.dtypes

class                    object
cap-diameter            float64
cap-shape                object
cap-surface              object
cap-color                object
does-bruise-or-bleed     object
gill-attachment          object
gill-spacing             object
gill-color               object
stem-height             float64
stem-width              float64
stem-root                object
stem-surface             object
stem-color               object
veil-type                object
veil-color               object
has-ring                 object
ring-type                object
spore-print-color        object
habitat                  object
season                   object
dtype: object

In [52]:
#Vérification données dupliquer
print(test.duplicated().sum())
print(train.duplicated().sum())
print("Pas de données dupliqués")

0
0
Pas de données dupliqués


In [36]:
#nombre de modalité qualitative
print(train[quali].nunique())

cap-shape               74
cap-surface             83
cap-color               78
does-bruise-or-bleed    26
gill-attachment         78
gill-spacing            48
gill-color              63
stem-root               38
stem-surface            60
stem-color              59
veil-type               22
veil-color              24
has-ring                23
ring-type               40
spore-print-color       32
habitat                 52
season                   4
dtype: int64

In [35]:
print("Nbr Donnnée manquante qualitative")
print(train[quali].isnull().sum())
print("Nbr Donnée manquante quantitative")
print(train[quanti].isnull().sum())

Nbr Donnnée manquante qualitative
cap-shape                    40
cap-surface              671023
cap-color                    12
does-bruise-or-bleed          8
gill-attachment          523936
gill-spacing            1258435
gill-color                   57
stem-root               2757023
stem-surface            1980861
stem-color                   38
veil-type               2957493
veil-color              2740947
has-ring                     24
ring-type                128880
spore-print-color       2849682
habitat                      45
season                        0
dtype: int64
Nbr Donnée manquante quantitative
cap-diameter    4
stem-height     0
stem-width      0
dtype: int64


In [51]:
#Pourcentage de données manquante
print("% Donnnée manquante qualitative dans train")
print(train[quali].isnull().sum()/train[quali].apply(len) *100)
print("% Donnnée manquante qualitative dans test")
print(test[quali].isnull().sum()/test[quali].apply(len) *100)


% Donnnée manquante qualitative dans train
cap-shape                0.001283
cap-surface             21.528227
cap-color                0.000385
does-bruise-or-bleed     0.000257
gill-attachment         16.809280
gill-spacing            40.373988
gill-color               0.001829
stem-root               88.452732
stem-surface            63.551362
stem-color               0.001219
veil-type               94.884350
veil-color              87.936970
has-ring                 0.000770
ring-type                4.134818
spore-print-color       91.425482
habitat                  0.001444
season                   0.000000
dtype: float64
% Donnnée manquante qualitative dans test
cap-shape                0.001492
cap-surface             21.506821
cap-color                0.000626
does-bruise-or-bleed     0.000481
gill-attachment         16.834796
gill-spacing            40.404694
gill-color               0.002358
stem-root               88.452543
stem-surface            63.595327
stem-color      

In [45]:
#Toute les lignes on au moins une donnée manquante
min(train.isnull().sum(axis=1))

1

## Stratégie de remplacement des données manquante
### Données quantitative
On utilise SimpleImputer (deux strategy: mean et median) ou KNNImputer.  
SimpleImputer(strategy=mean) si peux d'influence de valeur abhérente.  
SimpleImputer(strategy=median) si forte influence de valeur abhérente. 
KNNImputer ou on choisit le nombre de voisins ou faire la moyenne.  
### Données qualitative
On utilise SimpleImputer ou on prédit les données manquantes avec un KNNClassifier.   
SimpleImputer(strategy=most frequent) si peux de donnée manquante.  
Remplacer par une chaine de caractère "Missing".  
KNNClassifier entrainer sur les données complete et on prédit les données manquantes  

#### Méthode Brut: quanti SimpleImputer mean , quali SimpleImputer most_frequent

In [79]:
#pipeline pour transformer les colonnes
pipe_brut_quanti=make_pipeline(SimpleImputer(strategy='mean'),StandardScaler())
pie_brut_quali=make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore'))

#
ct_brut1=make_column_transformer([pipe_brut_quanti,quanti],[pie_brut_quali,quali],verbose_feature_names_out=False)

pipe_brut1=make_pipeline(ct_brut1,LogisticRegression(max_iter=1000))
pipe_brut1

,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [80]:
pipe_brut1.fit(train_x,train_y)

,steps,"[('columntransformer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


In [82]:
out=pd.DataFrame(pipe_brut1.predict(test),index=test.index,columns=['class'])
out
#out.to_csv('result_brut.csv')

,class
id,
3116945,e
3116946,p
3116947,p
3116948,p
3116949,e
...,...
5194904,p
5194905,e
5194906,p


# Score: 0.66880